# 4. Modeling

## 4.1 Strategy
We will train two distinct models to cover different threat scenarios:
1. **XGBoost (Supervised)**: For known attacks. To handle the 2.18% class imbalance, we will use `scale_pos_weight`.
2. **Isolation Forest (Unsupervised)**: For zero-day anomalies. It will be trained *only* on BENIGN traffic from the training set to learn the baseline behavior.

## 4.2 Artifacts
The trained models will be saved to `../models/` for deployment in the FastAPI backend.

In [1]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.ensemble import IsolationForest
from xgboost import XGBClassifier

# 1. Charger les données préparées
print("Chargement des données...")
X_train = pd.read_parquet("../data/processed/X_train.parquet")
X_test = pd.read_parquet("../data/processed/X_test.parquet")
y_train = pd.read_parquet("../data/processed/y_train.parquet")['Target']
y_test = pd.read_parquet("../data/processed/y_test.parquet")['Target']

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

Chargement des données...
Train shape: (79928, 78), Test shape: (19982, 78)


In [2]:
# 2. Calcul du scale_pos_weight (ratio de négatifs / positifs)
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight = neg_count / pos_count

print(f"Scale Pos Weight (Négatifs / Positifs) : {scale_pos_weight:.2f}")

# 3. Entraînement XGBoost
print("Entraînement du XGBoost...")
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight, # Gestion du déséquilibre
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)
print(" XGBoost entraîné !")

Scale Pos Weight (Négatifs / Positifs) : 44.83
Entraînement du XGBoost...
 XGBoost entraîné !


In [3]:
# 4. Entraînement Isolation Forest (sur le trafic Benign uniquement)
print("Entraînement de l'Isolation Forest...")

# On filtre le train set pour ne garder que le trafic BENIGN (Target == 0)
X_train_benign = X_train[y_train == 0]

iso_forest = IsolationForest(
    n_estimators=100,
    contamination=0.02, # On s'attend à environ 2% d'anomalies
    random_state=42,
    n_jobs=-1
)

iso_forest.fit(X_train_benign)
print(" Isolation Forest entraîné !")

Entraînement de l'Isolation Forest...
 Isolation Forest entraîné !


In [4]:
# 5. Sauvegarde des modèles pour le Backend
os.makedirs("../models", exist_ok=True)

joblib.dump(xgb_model, "../models/xgb_nids.joblib")
joblib.dump(iso_forest, "../models/iso_forest_nids.joblib")

print("✅ Modèles sauvegardés dans ../models/")

✅ Modèles sauvegardés dans ../models/
